# Getting started with py3spread

[3spread](https://3spread.com) serves machine-readable SEC filing data.
This notebook covers the basics: connecting, listing filings, pagination,
and error handling.

API keys are free, always. Get one at
[3spread.com/auth/signup](https://3spread.com/auth/signup) and put it in the
`THREESPREAD_API_KEY` environment variable before starting Jupyter.

```
pip install py3spread
```

In [1]:
from py3spread import Client

client = Client()  # reads THREESPREAD_API_KEY from the environment
client.health()

{'status': 'healthy',
 'service': '3spread API',
 'version': '0.1.0',
 'timestamp': '2026-07-02T22:31:01.918644Z',
 'checks': {'data': True, 'auth': True, 'gateway': True}}

## One schema for every filer

On EDGAR each of these filings is its own XML document with per-filer
quirks. Here it is one call and one set of labels, whether the insider
works at a mega cap, a meme stock, a soda company, or a pickup truck
maker:

In [3]:
import datetime as dt

import pandas as pd

window = dict(
    transaction_start=str(dt.date.today() - dt.timedelta(days=365)),
    transaction_end=str(dt.date.today()),
)

rows = []
for ticker in ["AAPL", "GME", "KO", "PLTR", "F"]:
    rows += client.insiders.transactions(
        issuer_ticker=ticker, transaction_kind="nonderiv", limit=1, **window
    )["data"]

pd.DataFrame(rows)[[
    "issuer_trading_symbol", "filer_name", "transaction_date", "transaction_code",
    "transaction_shares", "transaction_price_per_share", "transaction_total_value",
]]

,issuer_trading_symbol,filer_name,transaction_date,transaction_code,transaction_shares,transaction_price_per_share,transaction_total_value
0,AAPL,Borders Ben,2026-06-16,S,116.00000000,295.14000000,34236.2400000000000000
1,GME,Robinson Mark Haymond,2026-04-13,S,3912.00000000,23.18900000,90715.3680000000000000
2,KO,MANN JENNIFER K,2026-06-10,S,23984.00000000,83.41370000,2000594.1808000000000000
3,PLTR,Moore Alexander D.,2026-06-15,S,900.00000000,130.47830000,117430.4700000000000000
4,F,THORNTON JOHN L,2026-06-23,P,10600.00000000,14.04530000,148880.1800000000000000


Same columns, same types, same meaning, for every filer in the database.
And every row traces back to the raw filing it came from:

In [4]:
filing = client.insiders.list(ticker="AAPL", limit=1)["data"][0]
print("parsed by 3spread: ", filing["filing_id"])
print("raw source on EDGAR:", filing["source_url"])

parsed by 3spread:  320193_0001140361-26-025622
raw source on EDGAR: https://www.sec.gov/Archives/edgar/data/320193/000114036126025622/0001140361-26-025622-index.htm


## List filings

`filings` is the master index across every filing family. Windowed list
endpoints need at least one identity filter (cik or ticker) or a fully
bounded date window.

In [5]:
page = client.filings.list(ticker="AAPL", limit=5)
for f in page["data"]:
    print(f"{f['accepted_time']}  form {f['form_type']:<6} {f['filer_name']}")

2026-06-17T00:00:00  form 4      Apple Inc.
2026-06-17T00:00:00  form 4      Apple Inc.
2026-05-12T00:00:00  form 4      Apple Inc.
2026-05-08T00:00:00  form 4      Apple Inc.
2026-05-06T00:00:00  form 144    Apple Inc.


## Pagination

`list()` returns one page. The `iter()` variants follow the pagination
cursor for you and yield rows, so you can process large result sets without
thinking about pages.

In [6]:
import itertools
from collections import Counter

forms = Counter(
    f["form_type"] for f in itertools.islice(client.filings.iter(ticker="AAPL"), 100)
)
forms.most_common()

[('4', 74), ('144', 22), ('3', 4)]

## Errors

HTTP errors raise typed exceptions carrying `status_code`, `code`, and
`request_id`. Rate limits (429) and transient server errors (502, 503) are
retried automatically.

In [7]:
from py3spread import WindowTooWideError

try:
    client.filings.list(accepted_start="2024-01-01", accepted_end="2024-06-01")
except WindowTooWideError as e:
    print(f"{e.code}: {e.message[:120]}")

WINDOW_TOO_WIDE: Requested window (152d) exceeds the maximum (3d) for this endpoint. Add a cik/ticker filter to widen the allowed window,


## Where to go next

- `insider_analysis.ipynb`: insider transactions with pandas
- `institutional_13f.ipynb`: 13F holdings analysis
- `fund_overlap.ipynb`: portfolio overlap between two managers
- `company_360.ipynb`: everything 3spread knows about one company
- `insider_vs_institutions.ipynb`: do insiders and institutions agree?
- `activist_radar.ipynb`: every new 13D stake across the market
- [API reference](https://3spread.com/docs)